In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/pre_cell_2.pickle

trying: ['load_lineitem']
me:  1
trying: ['load_part']
me:  2
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1
trying: ['part']
me:  2
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_part
Declaring variable part


In [4]:
%%RecordEvent
%%time
### cell 2 ###

SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# Precompute lineitem columns to cut down __getitem__ calls
lq_line = lineitem["L_QUANTITY"]
si_line = lineitem["L_SHIPINSTRUCT"]
sm_line = lineitem["L_SHIPMODE"]

fline = lineitem[
    lq_line.between(4, 36)
    & (si_line == "DELIVER IN PERSON")
    & sm_line.isin(SHIPMODES)
]

# Precompute part columns
pb_part = part["P_BRAND"]
pc_part = part["P_CONTAINER"]
ps_part = part["P_SIZE"]

fpart = part[
    ((pb_part == "Brand#31") & pc_part.isin(SM_SMALL) & ps_part.between(1, 5))
    | ((pb_part == "Brand#43") & pc_part.isin(MED_CONTAINER) & ps_part.between(1, 10))
    | ((pb_part == "Brand#43") & pc_part.isin(LG_CONTAINER) & ps_part.between(1, 15))
]

# Join
df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# Precompute merged columns
lq = df["L_QUANTITY"]
pb = df["P_BRAND"]
pc = df["P_CONTAINER"]

# Final filter (size checks for Brand#31 guaranteed by fpart)
mask_brand31 = (pb == "Brand#31") & lq.between(4, 14)
mask_brand43 = (pb == "Brand#43") & (
    (pc.isin(MED_CONTAINER) & lq.between(15, 25))
    | (pc.isin(LG_CONTAINER)  & lq.between(26, 36))
)
df = df[mask_brand31 | mask_brand43]

# Compute total on GPU
total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()

CPU times: user 86.8 ms, sys: 12.7 ms, total: 99.5 ms
Wall time: 105 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_3.pickle

migration speed (bps): 784520092.6990062
---------------------------
variables to migrate:
pb 48
fline 48
sm_line 48
load_part 144
pc_part 48
pd 72
total 32
lq 48
part 48
df 48
load_lineitem 144
MED_CONTAINER 88
lineitem 48
si_line 48
tpch_parent 54
LG_CONTAINER 88
pb_part 48
mask_brand43 48
ps_part 48
pc 48
SM_SMALL 88
mask_brand31 48
DATA_ROOT 80
STORAGE_OPTS 64
SHIPMODES 72
lq_line 48
fpart 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['SHIPMODES', 'df', 'si_line', 'total', 'mask_brand43', 'ps_part', 'LG_CONTAINER', 'fline', 'MED_CONTAINER', 'mask_brand31', 'pc_part', 'sm_line', 'pb_part', 'SM_SMALL', 'lq_line', 'pc', 'fpart', 'pb', 'lq']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q19/opt_cell_exec_info_2_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/post_cell_2.pickle
assert total_opt == total

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['LGBOX']
me:  3
trying: ['SMCASE']
me:  3
trying: ['Brand31']
me:  3
trying: ['lsel']
me:  3
trying: ['total']
me:  3
trying: ['LGCASE']
me:  3
trying: ['LGPKG']
me:  3
trying: ['SMPACK']
me:  3
trying: ['jn']
me:  3
trying: ['load_part']
me:  2
trying: ['pd']
me:  0
trying: ['part']
me:  2
trying: ['fpart']
me:  3
trying: ['jnsel']
me:  3
trying: ['flineitem']
me:  3
trying: ['MEDPKG']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['Brand43']
me:  3
trying: ['MEDBOX']
me:  3
trying: ['lineitem']


me:  1
trying: ['LGPACK']
me:  3
trying: ['DELIVERINPERSON']
me:  3
trying: ['SMPKG']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['MEDPACK']
me:  3
trying: ['psel']
me:  3
trying: ['SMBOX']
me:  3
trying: ['AIR']
me:  3
trying: ['MEDBAG']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['AIRREG']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orig_output']
me:  4


Declaring variable pd
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_part
Declaring variable part
Declaring variable LGBOX
Declaring variable SMCASE
Declaring variable Brand31
Declaring variable lsel
Declaring variable total
Declaring variable LGCASE
Declaring variable LGPKG
Declaring variable SMPACK
Declaring variable jn
Declaring variable fpart
Declaring variable jnsel
Declaring variable flineitem
Declaring variable MEDPKG
Declaring variable Brand43
Declaring variable MEDBOX
Declaring variable LGPACK
Declaring variable DELIVERINPERSON
Declaring variable SMPKG
Declaring variable MEDPACK
Declaring variable psel
Declaring variable SMBOX
Declaring variable AIR
Declaring variable MEDBAG
Declaring variable AIRREG
Declaring variable orig_output
